In [9]:
import pandas as pd
import sqlite3

In [10]:
connection = sqlite3.connect('../../data/checking-logs.sqlite')

In [11]:
schema_test = pd.read_sql("PRAGMA table_info(test);", connection)
schema_test

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


In [12]:
df_test = pd.read_sql('SELECT * FROM test LIMIT 10;', connection)
df_test

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


In [13]:
query_min = '''
SELECT 
       MIN((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24) AS min_dif
FROM test AS t
JOIN deadlines AS d ON t.labname = d.labs
WHERE t.labname != 'project1';
'''

df_min = pd.read_sql(query_min, connection)
print(df_min)

    min_dif
0  2.867236


In [14]:
query_max = '''
SELECT 
       MAX((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24) AS max_dif
FROM test AS t
JOIN deadlines AS d ON t.labname = d.labs
WHERE t.labname != 'project1';
'''

df_max = pd.read_sql(query_max, connection)
print(df_max)

     max_dif
0  202.38473


In [15]:
query_avg = '''
SELECT AVG((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24) AS avg_diff
FROM test AS t
JOIN deadlines AS d ON t.labname = d.labs
WHERE t.labname != 'project1'
'''

df_avg = pd.read_sql(query_avg, connection)
print(df_avg)

    avg_diff
0  89.687686


In [16]:
query_views_diff = '''
SELECT t.uid, 
       AVG((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24) AS avg_diff, 
       COUNT(p.datetime) AS pageviews
FROM test t
JOIN deadlines d ON t.labname = d.labs
JOIN pageviews p ON t.uid = p.uid
WHERE t.labname != 'project1'
GROUP BY t.uid
'''

views_diff = pd.read_sql(query_views_diff, connection)
print(views_diff)

correlation = views_diff[['avg_diff', 'pageviews']].corr()
print(correlation)

        uid    avg_diff  pageviews
0    user_1   65.119644        140
1   user_10   75.242310        445
2   user_14  159.568696        429
3   user_17   62.207514        235
4   user_18    6.367907          9
5   user_19   99.440298         64
6   user_21   96.111042         40
7   user_25   93.474751        895
8   user_28   86.793652        745
9    user_3  105.738041       1585
10  user_30  145.528546         12
           avg_diff  pageviews
avg_diff   1.000000   0.185042
pageviews  0.185042   1.000000


In [17]:
connection.close()